# 01 — Umami Analytics Ingestion

Validate the Umami Cloud REST API integration:
- Connect with API key
- Fetch website stats, pageviews, metrics, events
- Parse into our `Insights` model
- Save to local state (mock S3)

In [1]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import os
prefix = os.getenv("S3_STATE_PREFIX", "growth-agent/")
if prefix == "growth-agent/":
    raise RuntimeError(
        f"S3_STATE_PREFIX is {prefix!r} — this is the PRODUCTION prefix. "
        "Set S3_STATE_PREFIX=growth-agent-dev/ in .env before running notebooks."
    )
print(f"Using S3 prefix: {prefix}  ✓")


Using S3 prefix: growth-agent-dev/  ✓


In [2]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env', override=True)

UMAMI_API_KEY = os.getenv('UMAMI_API_KEY')
UMAMI_WEBSITE_ID = os.getenv('UMAMI_WEBSITE_ID', 'e41ae7d9-a536-426d-b40e-f2488b11bf95')

print(f'API Key loaded: {"yes" if UMAMI_API_KEY else "NO — create .env from .env.example"}')
print(f'Website ID: {UMAMI_WEBSITE_ID}')

API Key loaded: yes
Website ID: e41ae7d9-a536-426d-b40e-f2488b11bf95


## 1. Connect to Umami Cloud API

Umami Cloud base URL: `https://api.umami.is/v1`  
Auth: `x-umami-api-key` header  
Rate limit: 50 calls / 15 seconds

In [3]:
from agent.umami_client import UmamiClient, ms_timestamp

umami = UmamiClient(api_key=UMAMI_API_KEY, website_id=UMAMI_WEBSITE_ID)

# Test connection: get active users
active = umami.get_active()
print(f'Active users right now: {active}')

Active users right now: {'visitors': 0}


## 2. Fetch Website Stats (last 7 days)

In [4]:
start = ms_timestamp(days_ago=7)
end = ms_timestamp(days_ago=0)

stats = umami.get_stats(start_at=start, end_at=end)
print('Website stats (7 days):')
for k, v in stats.items():
    if k != 'comparison':
        print(f'  {k}: {v}')

Website stats (7 days):
  pageviews: 74
  visitors: 54
  visits: 55
  bounces: 51
  totaltime: 122


## 3. Top Pages

In [5]:
top_pages = umami.get_metrics(start_at=start, end_at=end, metric_type='path', limit=15)
print('Top pages:')
for page in top_pages:
    print(f'  {page["y"]:>5} visitors — {page["x"]}')

Top pages:
     21 visitors — /
      6 visitors — /amo/13/
      5 visitors — /blog/
      5 visitors — /amo/10
      3 visitors — /amo/11
      3 visitors — /de/blog/
      3 visitors — /blog/29/
      2 visitors — /blog/27/
      2 visitors — /de/blog/27/
      1 visitors — /quantum/qml/
      1 visitors — /quantum/amo/18/
      1 visitors — /de/quantum/amo/5/
      1 visitors — /blog/12/
      1 visitors — /de/quantum/amo/6/
      1 visitors — /quantum/hardware/4/


## 4. Top Referrers

In [6]:
referrers = umami.get_metrics(start_at=start, end_at=end, metric_type='referrer', limit=15)
print('Top referrers:')
for ref in referrers:
    print(f'  {ref["y"]:>5} visitors — {ref["x"]}')

Top referrers:
      2 visitors — fretchen.eu
      1 visitors — go.bsky.app


## 5. Events (tracked funnels)

In [7]:
events = umami.get_metrics(start_at=start, end_at=end, metric_type='event', limit=30)
print('Tracked events:')
for event in events:
    print(f'  {event["y"]:>5}x — {event["x"]}')

Tracked events:
      1x — imagegen-connect-hover
      1x — wallet-dropdown-close
      1x — wallet-dropdown-open
      1x — blog-support-button-hover


## 6. Pageviews Over Time

In [8]:
pageviews = umami.get_pageviews(start_at=start, end_at=end, unit='day')
print('Daily pageviews:')
for pv in pageviews.get('pageviews', []):
    print(f'  {pv["x"][:10]} — {pv["y"]} views')

print('\nDaily sessions:')
for s in pageviews.get('sessions', []):
    print(f'  {s["x"][:10]} — {s["y"]} sessions')

Daily pageviews:
  2026-05-30 — 10 views
  2026-05-31 — 1 views
  2026-06-01 — 6 views
  2026-06-02 — 21 views
  2026-06-03 — 7 views
  2026-06-04 — 7 views
  2026-06-05 — 22 views

Daily sessions:
  2026-05-30 — 10 sessions
  2026-05-31 — 1 sessions
  2026-06-01 — 6 sessions
  2026-06-02 — 8 sessions
  2026-06-03 — 7 sessions
  2026-06-04 — 5 sessions
  2026-06-05 — 17 sessions


In [9]:
from datetime import datetime
from agent.models import Insights, WebsiteAnalytics

insights = Insights(
    website_analytics=WebsiteAnalytics(
        pageviews=stats.get('pageviews', 0),
        visitors=stats.get('visitors', 0),
        visits=stats.get('visits', 0),
        bounces=stats.get('bounces', 0),
        totaltime=stats.get('totaltime', 0),
        top_pages=top_pages[:10],
        top_referrers=referrers[:10],
        top_events=events[:10],
    ),
    last_analysis=datetime.now(),
)

print(f'Insights parsed: {insights.website_analytics.visitors} visitors, '
      f'{len(insights.website_analytics.top_pages)} top pages')


Insights parsed: 54 visitors, 10 top pages


## 7. Save to S3 State

In [10]:
from agent.storage import S3Storage

store = S3Storage(
    bucket=os.getenv('S3_BUCKET'),
    prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
    access_key=os.getenv('SCW_ACCESS_KEY'),
    secret_key=os.getenv('SCW_SECRET_KEY'),
)
store.write('insights.json', insights)

# Verify round-trip
loaded = store.read('insights.json')
print(f'Saved and loaded insights.json ({len(str(loaded))} chars)')
print(f'Top page: {loaded["website_analytics"]["top_pages"][0] if loaded["website_analytics"]["top_pages"] else "none"}')

Saved and loaded insights.json (746 chars)
Top page: {'x': '/', 'y': 21}


In [11]:
umami.close()
print('Done — Umami ingestion validated.')

Done — Umami ingestion validated.


## 8. Per-post Engagement Metrics

Copy the production `content_queue.json` into the dev prefix (read-only from prod), then run `_collect_post_metrics()` to validate the new metrics collection code.

Requires `S3_STATE_PREFIX_PROD` env var set to the production prefix (e.g. `growth-agent/`).

In [12]:
# Copy prod content_queue.json → dev prefix (prod is read-only here)
import boto3

prod_prefix = os.environ["S3_STATE_PREFIX_PROD"]  # e.g. "growth-agent/"
dev_prefix = os.getenv("S3_STATE_PREFIX", "growth-agent-dev/")
bucket = os.environ["S3_BUCKET"]

s3_raw = boto3.client(
    "s3",
    region_name="nl-ams",
    endpoint_url="https://s3.nl-ams.scw.cloud",
    aws_access_key_id=os.environ["SCW_ACCESS_KEY"],
    aws_secret_access_key=os.environ["SCW_SECRET_KEY"],
)
s3_raw.copy_object(
    Bucket=bucket,
    CopySource={"Bucket": bucket, "Key": prod_prefix + "content_queue.json"},
    Key=dev_prefix + "content_queue.json",
)
print(f"Copied {prod_prefix}content_queue.json → {dev_prefix}content_queue.json")


Copied growth-agent/content_queue.json → growth-agent-dev/content_queue.json


In [13]:
# Run metrics collection against the copied queue
from agent.nodes.ingest import _collect_post_metrics

_collect_post_metrics(store)
print("Done — performance.json written to dev prefix")


Done — performance.json written to dev prefix


In [15]:
# Display results
from agent.storage import load_model
from agent.models import Performance

perf = load_model(store, "performance.json", Performance)
print(f"Posts with metrics: {len(perf.posts)}")
for p in perf.posts:
    print(f"  [{p.channel:8}] ❤️{p.favourites:3}  🔁{p.reblogs:3}  💬{p.replies:3}  — {p.id[:24]}…")


Posts with metrics: 64
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  1  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  1  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20251…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  1  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  1  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  0  🔁  0  💬  0  — recovered_mastodon_20260…
  [mastodon] ❤️  